In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pwd

/content


In [3]:
%cd '/content/drive/MyDrive/Projects/Water quality'

/content/drive/MyDrive/Projects/Water quality


In [4]:
!pwd

/content/drive/MyDrive/Projects/Water quality


# Importing data

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df_raw = pd.read_csv('water_potability.csv')

In [7]:
df_raw

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0
...,...,...,...,...,...,...,...,...,...,...
3271,4.668102,193.681735,47580.991603,7.166639,359.948574,526.424171,13.894419,66.687695,4.435821,1
3272,7.808856,193.553212,17329.802160,8.061362,NaN,392.449580,19.903225,NaN,2.798243,1
3273,9.419510,175.762646,33155.578218,7.350233,NaN,432.044783,11.039070,69.845400,3.298875,1
3274,5.126763,230.603758,11983.869376,6.303357,NaN,402.883113,11.168946,77.488213,4.708658,1


In [8]:
target_feat = 'Potability'

## Train validation test split

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df_raw.drop(target_feat, axis='columns'), df_raw[target_feat], test_size=0.2, random_state=2)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

In [10]:
y_train

,Potability
3216,1
3050,0
739,1
2100,0
2035,1
...,...
2347,1
1608,1
2541,0
2575,0


In [11]:
X_train

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
3216,5.609235,216.122673,14825.934502,7.001788,355.211678,330.092674,9.779518,41.249977,3.224676
3050,8.479593,200.765679,14901.808144,7.379506,NaN,485.093167,16.806347,53.134170,4.221266
739,6.772157,196.900098,13790.296202,9.575053,NaN,347.588375,17.705727,68.379815,3.827256
2100,4.894278,184.552715,10922.541994,7.461703,352.830222,338.681069,21.624718,91.007934,3.594991
2035,5.763773,183.073629,22025.696606,8.952896,NaN,376.581389,10.547398,63.299757,3.253200
...,...,...,...,...,...,...,...,...,...
2347,5.429335,183.439383,15265.407564,5.714731,394.001195,446.879149,17.581557,50.266951,3.081736
1608,6.919726,194.859000,35558.731647,6.371978,299.786711,394.829326,11.983961,69.853135,4.120828
2541,5.735724,158.318741,25363.016594,7.728601,377.543291,568.304671,13.626624,75.952337,4.732954
2575,6.914868,206.249937,10343.378848,7.771206,324.383170,521.320673,16.173730,68.246945,2.988611


In [12]:
X_test

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
2013,NaN,192.742275,32349.102378,8.750630,272.085057,581.981191,13.583036,64.436046,3.952594
861,NaN,184.750698,28499.107328,6.550031,330.719549,381.501604,11.044405,62.014822,3.403270
599,NaN,205.638790,39742.970329,4.660528,323.956492,509.546419,11.674850,55.042679,3.916746
1975,6.804796,223.995701,17196.054711,7.112622,374.309131,344.718314,15.457383,60.151346,4.847313
20,NaN,227.435048,22305.567414,10.333918,NaN,554.820086,16.331693,45.382815,4.133423
...,...,...,...,...,...,...,...,...,...
3217,NaN,98.367915,28415.575832,10.558950,296.843208,505.240269,12.882614,85.329955,4.119087
3078,11.390543,145.341937,16175.221944,6.613865,NaN,553.488923,12.091189,69.736231,2.277359
1561,6.227827,169.926222,22550.492264,4.973593,297.404497,460.415386,20.789592,93.300733,4.007347
699,7.376449,168.750007,37191.114141,4.184943,NaN,439.312853,15.900985,63.334895,5.767355


In [13]:
y_train.value_counts()/len(y_train)

,count
Potability,
0,0.604962
1,0.395038


In [14]:
y_test.value_counts()/len(y_test)

,count
Potability,
0,0.655488
1,0.344512


In [15]:
y_val.value_counts()/len(y_val)

,count
Potability,
0,0.603659
1,0.396341


Imputing null values with mean

In [16]:
for col in X_train.columns:
  X_train[col] = X_train[col].fillna(X_train[col].mean())
  X_val[col] = X_val[col].fillna(X_val[col].mean())
  X_test[col] = X_test[col].fillna(X_test[col].mean())

In [17]:
df_raw.isnull().sum()

,0
ph,491
Hardness,0
Solids,0
Chloramines,0
Sulfate,781
Conductivity,0
Organic_carbon,0
Trihalomethanes,162
Turbidity,0
Potability,0


In [18]:
X_train.isnull().sum()

,0
ph,0
Hardness,0
Solids,0
Chloramines,0
Sulfate,0
Conductivity,0
Organic_carbon,0
Trihalomethanes,0
Turbidity,0


In [19]:
X_val.isnull().sum()

,0
ph,0
Hardness,0
Solids,0
Chloramines,0
Sulfate,0
Conductivity,0
Organic_carbon,0
Trihalomethanes,0
Turbidity,0


In [20]:
X_test.isnull().sum()

,0
ph,0
Hardness,0
Solids,0
Chloramines,0
Sulfate,0
Conductivity,0
Organic_carbon,0
Trihalomethanes,0
Turbidity,0


#### Min max scaler

In [26]:
x_min, x_max = X_train.min(), X_train.max()

In [27]:
X_train = (X_train - x_min) / (x_max - x_min)

In [28]:
X_val = (X_val - x_min) / (x_max - x_min)
X_test = (X_test - x_min) / (x_max - x_min)

In [29]:
X_train

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
3216,0.400660,0.584920,0.238153,0.520531,0.650894,0.282128,0.305543,0.328666,0.335541
3050,0.605685,0.521942,0.239398,0.550098,0.587751,0.576390,0.588806,0.425080,0.523968
739,0.483725,0.506090,0.221149,0.721961,0.587751,0.315343,0.625062,0.548765,0.449472
2100,0.349591,0.455454,0.174064,0.556532,0.644042,0.298433,0.783043,0.732342,0.405557
2035,0.411698,0.449388,0.356363,0.673260,0.587751,0.370385,0.336498,0.507551,0.340934
...,...,...,...,...,...,...,...,...,...
2347,0.387810,0.450888,0.245368,0.419783,0.762506,0.503843,0.620056,0.401818,0.308515
1608,0.494266,0.497719,0.578558,0.471231,0.491416,0.405028,0.394408,0.560717,0.504978
2541,0.409695,0.347869,0.411158,0.577425,0.715151,0.734364,0.460626,0.610199,0.620713
2575,0.493919,0.544433,0.164555,0.580760,0.562189,0.645167,0.563305,0.547687,0.290908


In [30]:
X_val

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
3114,0.497970,0.461007,0.453797,0.537183,0.635162,0.441551,0.712828,0.287526,0.335857
1722,0.540466,0.540933,0.500815,0.584316,0.647914,0.405917,0.398894,0.511347,0.345083
2478,0.790675,0.749284,0.709534,0.289285,0.598064,0.478397,0.412283,0.446163,0.648613
401,0.614028,0.475837,0.233113,0.373552,0.845919,0.316203,0.611047,0.519389,0.391356
227,0.505984,0.161282,0.135607,0.302018,0.304577,0.480812,0.854588,0.757917,0.460896
...,...,...,...,...,...,...,...,...,...
1196,0.506245,0.571742,0.226265,0.685752,0.747505,0.295604,0.258093,0.599079,0.411162
3129,0.456993,0.723724,0.356878,0.355438,0.709256,0.290058,0.481739,0.781201,0.474294
263,0.941100,-0.106872,0.310592,0.669669,0.708257,0.605157,0.398436,0.519389,0.502349
1853,0.284163,0.358121,0.231714,0.630833,0.524531,0.442046,0.748648,0.629028,0.435161


In [31]:
X_test

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
2013,0.506161,0.489039,0.525860,0.657427,0.411709,0.760329,0.458869,0.516770,0.473170
861,0.506161,0.456265,0.462648,0.485169,0.580422,0.379726,0.356533,0.497127,0.369308
599,0.506161,0.541927,0.647257,0.337263,0.560962,0.622814,0.381947,0.440563,0.466392
1975,0.486057,0.617207,0.277067,0.529207,0.705845,0.309894,0.534427,0.482009,0.642336
20,0.506161,0.631312,0.360958,0.781363,0.592346,0.708764,0.569672,0.362194,0.507359
...,...,...,...,...,...,...,...,...,...
3217,0.506161,0.102014,0.461277,0.798978,0.482947,0.614639,0.430634,0.686278,0.504649
3078,0.813610,0.294652,0.260306,0.490166,0.592346,0.706237,0.398730,0.559769,0.156430
1561,0.444845,0.395471,0.364980,0.361769,0.484562,0.529541,0.749378,0.750943,0.483522
699,0.526889,0.390647,0.605359,0.300035,0.592346,0.489478,0.552310,0.507836,0.816289


## Logistic regression

In [21]:
from sklearn.linear_model import LogisticRegression

In [56]:
log_reg = LogisticRegression(max_iter=1000, class_weight = {0: 0.6, 1: 0.4})

In [57]:
log_reg

LogisticRegression(class_weight={0: 0.6, 1: 0.4}, max_iter=1000)

### Training

In [58]:
log_reg.fit(X_train, y_train)

LogisticRegression(class_weight={0: 0.6, 1: 0.4}, max_iter=1000)

In [59]:
log_reg.score(X_val, y_val)

0.6036585365853658

In [60]:
from sklearn.metrics import f1_score

In [61]:
y_pred_val = log_reg.predict(X_val)
f1 = f1_score(y_val, y_pred_val, average='macro')
print(f"Sklearn Macro F1: {f1:.4f}")

Sklearn Macro F1: 0.3764


## WaterQualityModel MLP

In [64]:
import torch
import torch.nn as nn

class WaterQualityModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear_layer1 = nn.Linear(in_features=9, out_features=16, dtype=torch.float)
    self.relu = nn.ReLU()
    self.norm1 = nn.BatchNorm1d(16)
    self.linear_layer2 = nn.Linear(in_features=16, out_features=1, dtype=torch.float)

  def forward(self, x):
    z = self.linear_layer1(x)
    z = self.norm1(z)
    z = self.relu(z)
    z = self.linear_layer2(z)
    return z

In [67]:
X_train_tensor = torch.from_numpy(X_train.to_numpy()).type(torch.float)
y_train_tensor = torch.from_numpy(y_train.to_numpy()).type(torch.float)

X_val_tensor = torch.from_numpy(X_val.to_numpy()).type(torch.float)
y_val_tensor = torch.from_numpy(y_val.to_numpy()).type(torch.float)

X_test_tensor = torch.from_numpy(X_test.to_numpy()).type(torch.float)
y_test_tensor = torch.from_numpy(y_test.to_numpy()).type(torch.float)

In [65]:
model = WaterQualityModel()
model = torch.load('WaterQualityModel_retry.bin', weights_only=False)

In [69]:
def accuracy(y_pred, y_true):
  return torch.eq(torch.round(torch.sigmoid(y_pred)), y_true.unsqueeze(dim=1)).sum().item()/len(y_pred)

In [70]:
with torch.inference_mode():
  y_pred_val = torch.round(torch.sigmoid(model(X_val_tensor))).detach().cpu().numpy().squeeze()
  y_true = y_val_tensor.detach().cpu().numpy()
  f1 = f1_score(y_true, y_pred_val, average='macro')
  acc = accuracy(torch.from_numpy(y_pred_val.reshape((-1, 1))), torch.from_numpy(y_true))
  print(f"Sklearn Macro F1: {f1:.4f}")
  print(f"Accuracy : {acc:.4f}")

Sklearn Macro F1: 0.6402
Accuracy : 0.6860


**Observation**

Using a non-linear model like mlp, we are able to achieve far better f1 scores than a plain logistic regression model